In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mohdtahasayed2904/asdfg12345/Malware_Dataset.csv
/kaggle/input/datasets/mohdtahasayed2904/finetuning-continuation/adapter_config (3).json
/kaggle/input/datasets/mohdtahasayed2904/finetuning-continuation/adapter_model (6).safetensors
/kaggle/input/datasets/mohdtahasayed2904/finetuning-continuation/tokenizer (3).json
/kaggle/input/datasets/mohdtahasayed2904/finetuning-continuation/tokenizer_config (4).json
/kaggle/input/datasets/mohdtahasayed2904/finetuning-continuation/training_config (1).json
/kaggle/input/datasets/mohdtahasayed2904/finetuning-continuation/training_state.pt


In [3]:
# ================================================================
# CELL 1: DEEPSEEK-CODER MALWARE DATASET PREPARATION
# CSV -> VALIDATION -> SPLITTING -> PROMPTS -> MASKED TOKENIZATION
# ================================================================

import os
import glob
import json
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

# ================================================================
# 1. CONFIGURATION
# ================================================================

SEED = 42
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-base"
MAX_LENGTH = 4096

WORK_DIR = "/kaggle/working/deepseek_malware"

os.makedirs(WORK_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Configuration initialized.")


# ================================================================
# 2. AUTOMATICALLY LOCATE AND LOAD CSV
# ================================================================

csv_files = glob.glob(
    "/kaggle/input/**/*.csv",
    recursive=True
)

valid_files = []

for path in csv_files:
    try:
        columns = pd.read_csv(path, nrows=0).columns

        if {"SHA256", "Type"}.issubset(columns):
            valid_files.append(path)

    except Exception:
        continue

if not valid_files:
    raise FileNotFoundError(
        "No CSV containing SHA256 and Type was found."
    )

valid_files.sort(key=os.path.getsize, reverse=True)

CSV_PATH = valid_files[0]

print("\nSelected CSV:", CSV_PATH)

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)


# ================================================================
# 3. DATASET VALIDATION
# ================================================================

assert "SHA256" in df.columns
assert "Type" in df.columns

assert df["SHA256"].is_unique, "Duplicate SHA256 detected."

assert df.isnull().sum().sum() == 0, "Missing values detected."

FEATURE_COLUMNS = [
    col for col in df.columns
    if col not in ["SHA256", "Type"]
]

assert len(FEATURE_COLUMNS) == 318, (
    f"Expected 318 features, found {len(FEATURE_COLUMNS)}"
)

assert all(
    pd.api.types.is_numeric_dtype(df[col])
    for col in FEATURE_COLUMNS
), "Non-numeric feature detected."

df["Type"] = pd.to_numeric(
    df["Type"],
    errors="raise"
).astype(int)

LABEL_MAP = {
    0: "Benign",
    1: "RedLineStealer",
    2: "Downloader",
    3: "RAT",
    4: "BankingTrojan",
    5: "SnakeKeyLogger",
    6: "Spyware"
}

assert set(df["Type"].unique()) == set(LABEL_MAP.keys())

print("\nDataset validation successful.")

print("Total samples:", len(df))
print("Total PE features:", len(FEATURE_COLUMNS))

print("\nClass distribution:")
print(df["Type"].value_counts().sort_index())


# ================================================================
# 4. FEATURE GROUPS
# ================================================================

API_FEATURES = FEATURE_COLUMNS[:200]
DLL_FEATURES = FEATURE_COLUMNS[200:227]
HEADER_FEATURES = FEATURE_COLUMNS[227:273]
SECTION_FEATURES = FEATURE_COLUMNS[273:318]

assert len(API_FEATURES) == 200
assert len(DLL_FEATURES) == 27
assert len(HEADER_FEATURES) == 46
assert len(SECTION_FEATURES) == 45

print("\nFeature groups:")
print("API Functions:", len(API_FEATURES))
print("DLL Imports:", len(DLL_FEATURES))
print("PE Header:", len(HEADER_FEATURES))
print("PE Sections:", len(SECTION_FEATURES))


# ================================================================
# 5. STRATIFIED TRAIN / VALIDATION / TEST SPLIT
# ================================================================

development_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["Type"]
)

train_df, validation_df = train_test_split(
    development_df,
    test_size=0.25,
    random_state=SEED,
    stratify=development_df["Type"]
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

assert len(train_df) == 17693
assert len(validation_df) == 5898
assert len(test_df) == 5898

# Verify no sample leakage.

assert set(train_df["SHA256"]).isdisjoint(
    validation_df["SHA256"]
)

assert set(train_df["SHA256"]).isdisjoint(
    test_df["SHA256"]
)

assert set(validation_df["SHA256"]).isdisjoint(
    test_df["SHA256"]
)

print("\nDataset split completed.")

print("Training:", len(train_df))
print("Validation:", len(validation_df))
print("Testing:", len(test_df))

# Save splits.

for name, split_df in [
    ("train", train_df),
    ("validation", validation_df),
    ("test", test_df)
]:

    split_df.to_csv(
        f"{WORK_DIR}/{name}.csv",
        index=False
    )


# ================================================================
# 6. LOAD DEEPSEEK TOKENIZER
# ================================================================

print("\nLoading DeepSeek tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded.")
print("EOS token:", tokenizer.eos_token)
print("EOS ID:", tokenizer.eos_token_id)


# ================================================================
# 7. PROMPT TEMPLATE
# ================================================================

SYSTEM_PROMPT = """You are a Windows PE malware classification assistant.

Classify the executable using only its provided static PE features.

Instructions:
1. Analyze all provided feature values.
2. Zero values are meaningful and must not be ignored.
3. Do not invent missing information.
4. Do not infer unsupported runtime behavior.
5. Select exactly one malware family.

Available classes:
Benign, RedLineStealer, Downloader, RAT,
BankingTrojan, SnakeKeyLogger, Spyware.
"""


def format_features(row, columns):

    return "\n".join(
        f"{col}={row[col]}"
        for col in columns
    )


def create_prompt(row):

    return f"""### SYSTEM:
{SYSTEM_PROMPT}

### USER:
Analyze the following Windows PE static feature profile.

<API_FUNCTIONS>
{format_features(row, API_FEATURES)}
</API_FUNCTIONS>

<DLL_IMPORTS>
{format_features(row, DLL_FEATURES)}
</DLL_IMPORTS>

<PE_HEADER>
{format_features(row, HEADER_FEATURES)}
</PE_HEADER>

<PE_SECTIONS>
{format_features(row, SECTION_FEATURES)}
</PE_SECTIONS>

Classify this executable into exactly one of the seven malware families.

### ASSISTANT:
"""


def create_completion(row):

    return (
        "Predicted Malware Family: "
        + LABEL_MAP[int(row["Type"])]
    )


# ================================================================
# 8. MASKED SFT TOKENIZATION
# ================================================================

def prepare_records(dataframe, split_name):

    records = []

    max_length_observed = 0

    for index, row in dataframe.iterrows():

        prompt = create_prompt(row)

        completion = create_completion(row)

        # Tokenize prompt separately.

        prompt_ids = tokenizer(
            prompt,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]

        # Tokenize assistant response separately.

        completion_ids = tokenizer(
            completion,
            add_special_tokens=False,
            truncation=False
        )["input_ids"]

        # Add supervised EOS token.

        completion_ids.append(tokenizer.eos_token_id)

        input_ids = prompt_ids + completion_ids

        # Mask prompt tokens.

        labels = (
            [-100] * len(prompt_ids)
            + completion_ids
        )

        if len(input_ids) > MAX_LENGTH:

            raise ValueError(
                f"{split_name} sample {index} exceeds "
                f"{MAX_LENGTH} tokens: {len(input_ids)}"
            )

        assert all(
            x == -100 for x in labels[:len(prompt_ids)]
        )

        assert labels[len(prompt_ids):] == completion_ids

        assert input_ids[-1] == tokenizer.eos_token_id

        records.append({
            "input_ids": input_ids,
            "labels": labels,
            "label_id": int(row["Type"]),
            "sha256": str(row["SHA256"])
        })

        max_length_observed = max(
            max_length_observed,
            len(input_ids)
        )

        if (index + 1) % 2000 == 0:

            print(
                f"{split_name}: "
                f"{index + 1}/{len(dataframe)} processed"
            )

    print(
        f"{split_name} completed | "
        f"Maximum length: {max_length_observed}"
    )

    return records


print("\nStarting tokenization...")

train_records = prepare_records(
    train_df, "Training"
)

validation_records = prepare_records(
    validation_df, "Validation"
)

test_records = prepare_records(
    test_df, "Testing"
)


# ================================================================
# 9. SAVE TOKENIZED DATASETS
# ================================================================

for name, records in [
    ("train", train_records),
    ("validation", validation_records),
    ("test", test_records)
]:

    output_path = f"{WORK_DIR}/{name}_tokenized.jsonl"

    with open(output_path, "w", encoding="utf-8") as f:

        for record in records:

            f.write(
                json.dumps(record) + "\n"
            )

    print(f"Saved: {output_path}")


# ================================================================
# 10. FINAL VERIFICATION
# ================================================================

print("\n" + "=" * 65)
print("CELL 1 COMPLETED SUCCESSFULLY")
print("=" * 65)

print("\nDataset sizes:")
print("Training:", len(train_records))
print("Validation:", len(validation_records))
print("Testing:", len(test_records))

print("\nMaximum sequence lengths:")

for name, records in [
    ("Training", train_records),
    ("Validation", validation_records),
    ("Testing", test_records)
]:

    print(
        name,
        max(len(x["input_ids"]) for x in records)
    )

sample = train_records[0]

masked_tokens = sum(
    x == -100 for x in sample["labels"]
)

supervised_tokens = sum(
    x != -100 for x in sample["labels"]
)

print("\nMasked SFT verification:")
print("Masked prompt tokens:", masked_tokens)
print("Supervised response tokens:", supervised_tokens)

print("\nDecoded assistant response:")

print(
    tokenizer.decode(
        sample["input_ids"][masked_tokens:],
        skip_special_tokens=False
    )
)

print("\nAll files saved in:")
print(WORK_DIR)

print("\nREADY FOR CELL 2: DEEPSEEK MODEL + LoRA SETUP")

Configuration initialized.

Selected CSV: /kaggle/input/datasets/mohdtahasayed2904/asdfg12345/Malware_Dataset.csv
Dataset shape: (29489, 320)

Dataset validation successful.
Total samples: 29489
Total PE features: 318

Class distribution:
Type
0    1877
1    5022
2    4643
3    4955
4    5076
5    4217
6    3699
Name: count, dtype: int64

Feature groups:
API Functions: 200
DLL Imports: 27
PE Header: 46
PE Sections: 45

Dataset split completed.
Training: 17693
Validation: 5898
Testing: 5898

Loading DeepSeek tokenizer...
Tokenizer loaded.
EOS token: <｜end▁of▁sentence｜>
EOS ID: 32014

Starting tokenization...
Training: 2000/17693 processed
Training: 4000/17693 processed
Training: 6000/17693 processed
Training: 8000/17693 processed
Training: 10000/17693 processed
Training: 12000/17693 processed
Training: 14000/17693 processed
Training: 16000/17693 processed
Training completed | Maximum length: 2963
Validation: 2000/5898 processed
Validation: 4000/5898 processed
Validation completed | Maxi

In [4]:
# ================================================================
# CELL 2: RESUME DEEPSEEK QLoRA TRAINING FROM CHECKPOINT 300
# TWO GPUs | CONTINUE FROM STEP 301 TO STEP 553
# ================================================================

import os
import sys
import json
import shutil
import subprocess
from pathlib import Path

import torch


# ================================================================
# 1. PATHS AND CONFIGURATION
# ================================================================

CHECKPOINT_INPUT_DIR = Path(
    "/kaggle/input/datasets/mohdtahasayed2904/finetuning-continuation"
)

CHECKPOINT_DIR = Path(
    "/kaggle/working/deepseek_resume_checkpoint"
)

WORK_DIR = Path(
    "/kaggle/working/deepseek_malware"
)

OUTPUT_DIR = Path(
    "/kaggle/working/deepseek_malware_resumed"
)

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-base"

CHECKPOINT_STEP = 300
TOTAL_STEPS = 553

GRAD_ACCUMULATION = 16
BATCH_SIZE = 1

LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01

WARMUP_STEPS = 16
MAX_LENGTH = 4096

NUM_GPUS = 2

SEED = 42

print("=" * 65)
print("DEEPSEEK CHECKPOINT RESUMPTION")
print("=" * 65)


# ================================================================
# 2. VERIFY DATASET FILES FROM CELL 1
# ================================================================

TRAIN_FILE = WORK_DIR / "train_tokenized.jsonl"
VALIDATION_FILE = WORK_DIR / "validation_tokenized.jsonl"
TEST_FILE = WORK_DIR / "test_tokenized.jsonl"

required_data_files = [
    TRAIN_FILE,
    VALIDATION_FILE,
    TEST_FILE
]

for file in required_data_files:
    if not file.exists():
        raise FileNotFoundError(
            f"Missing tokenized dataset: {file}\n"
            "Run Cell 1 before running this cell."
        )

print("\nTokenized datasets found.")

print("Training:", TRAIN_FILE)
print("Validation:", VALIDATION_FILE)
print("Testing:", TEST_FILE)


# ================================================================
# 3. COPY INDIVIDUALLY UPLOADED CHECKPOINT FILES
# ================================================================

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

FILE_MAPPING = {
    "adapter_config (3).json": "adapter_config.json",
    "adapter_model (6).safetensors": "adapter_model.safetensors",
    "tokenizer (3).json": "tokenizer.json",
    "tokenizer_config (4).json": "tokenizer_config.json",
    "training_config (1).json": "training_config.json",
    "training_state.pt": "training_state.pt"
}

print("\nCopying checkpoint files...")

for source_name, destination_name in FILE_MAPPING.items():

    source = CHECKPOINT_INPUT_DIR / source_name
    destination = CHECKPOINT_DIR / destination_name

    if not source.exists():
        raise FileNotFoundError(
            f"Checkpoint file not found: {source}"
        )

    shutil.copy2(source, destination)

    print("Copied:", source_name)


# ================================================================
# 4. VERIFY CHECKPOINT
# ================================================================

state_path = CHECKPOINT_DIR / "training_state.pt"

checkpoint_state = torch.load(
    state_path,
    map_location="cpu",
    weights_only=False
)

assert checkpoint_state["global_step"] == CHECKPOINT_STEP, (
    f"Expected step {CHECKPOINT_STEP}, "
    f"found {checkpoint_state['global_step']}"
)

assert "optimizer" in checkpoint_state
assert "scheduler" in checkpoint_state

assert (
    CHECKPOINT_DIR / "adapter_config.json"
).exists()

assert (
    CHECKPOINT_DIR / "adapter_model.safetensors"
).exists()

print("\nCheckpoint verification successful.")

print("Global step:", checkpoint_state["global_step"])

print(
    "Optimizer state entries:",
    len(checkpoint_state["optimizer"]["state"])
)

print("Scheduler state restored from checkpoint.")

del checkpoint_state

# ================================================================
# 5. PREPARE OUTPUT DIRECTORY
# ================================================================

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nOutput directory:", OUTPUT_DIR)


# ================================================================
# 6. CREATE DISTRIBUTED TRAINING SCRIPT
# ================================================================

SCRIPT_PATH = Path(
    "/kaggle/working/resume_deepseek_ddp.py"
)

script = r'''
import os
import json
import math
import random
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.distributed as dist

from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

from peft import (
    PeftModel,
    prepare_model_for_kbit_training
)

from torch.nn.parallel import DistributedDataParallel as DDP


# ================================================================
# CONFIGURATION
# ================================================================

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-base"

WORK_DIR = Path("/kaggle/working/deepseek_malware")

CHECKPOINT_DIR = Path(
    "/kaggle/working/deepseek_resume_checkpoint"
)

OUTPUT_DIR = Path(
    "/kaggle/working/deepseek_malware_resumed"
)

CHECKPOINT_STEP = 300
TOTAL_STEPS = 553

GRAD_ACCUMULATION = 16
BATCH_SIZE = 1

LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01

WARMUP_STEPS = 16

SEED = 42

TRAIN_FILE = WORK_DIR / "train_tokenized.jsonl"


# ================================================================
# DATASET
# ================================================================

class MalwareDataset(Dataset):

    def __init__(self, path):

        self.records = []

        with open(path, "r", encoding="utf-8") as f:

            for line in f:
                self.records.append(json.loads(line))

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        return self.records[index]


class MalwareCollator:

    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, batch):

        max_len = max(
            len(item["input_ids"])
            for item in batch
        )

        input_ids = []
        attention_mask = []
        labels = []

        for item in batch:

            ids = item["input_ids"]
            labs = item["labels"]

            padding_length = max_len - len(ids)

            input_ids.append(
                ids + [self.pad_token_id] * padding_length
            )

            attention_mask.append(
                [1] * len(ids) + [0] * padding_length
            )

            labels.append(
                labs + [-100] * padding_length
            )

        return {
            "input_ids": torch.tensor(
                input_ids,
                dtype=torch.long
            ),

            "attention_mask": torch.tensor(
                attention_mask,
                dtype=torch.long
            ),

            "labels": torch.tensor(
                labels,
                dtype=torch.long
            )
        }


# ================================================================
# SCHEDULER
# ================================================================

def lr_lambda(current_step):

    if current_step < WARMUP_STEPS:

        return float(current_step + 1) / float(WARMUP_STEPS)

    remaining = max(
        0,
        TOTAL_STEPS - current_step
    )

    return remaining / max(
        1,
        TOTAL_STEPS - WARMUP_STEPS
    )


# ================================================================
# CHECKPOINT SAVING
# ================================================================

def save_checkpoint(
    model,
    tokenizer,
    optimizer,
    scheduler,
    global_step,
    rank
):

    if rank != 0:
        return

    checkpoint_path = (
        OUTPUT_DIR / f"checkpoint-{global_step}"
    )

    checkpoint_path.mkdir(
        parents=True,
        exist_ok=True
    )

    model.module.save_pretrained(
        checkpoint_path
    )

    tokenizer.save_pretrained(
        checkpoint_path
    )

    torch.save(
        {
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "global_step": global_step
        },
        checkpoint_path / "training_state.pt"
    )

    print(
        f"\nCheckpoint saved at step {global_step}",
        flush=True
    )

    # Retain the latest three checkpoints.

    checkpoints = sorted(
        [
            p for p in OUTPUT_DIR.glob("checkpoint-*")
            if p.is_dir()
        ],
        key=lambda p: int(p.name.split("-")[-1])
    )

    while len(checkpoints) > 3:

        old_checkpoint = checkpoints.pop(0)

        shutil.rmtree(old_checkpoint)

        print(
            "Removed old checkpoint:",
            old_checkpoint.name,
            flush=True
        )


# ================================================================
# TRAINING WORKER
# ================================================================

def train_worker():

    dist.init_process_group(
        backend="nccl"
    )

    LOCAL_RANK = int(os.environ["LOCAL_RANK"])
    RANK = int(os.environ["RANK"])
    WORLD_SIZE = int(os.environ["WORLD_SIZE"])

    torch.cuda.set_device(LOCAL_RANK)

    device = torch.device(
        f"cuda:{LOCAL_RANK}"
    )

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    if RANK == 0:

        print("\nDistributed training initialized.")
        print("World size:", WORLD_SIZE)
        print("Starting from step:", CHECKPOINT_STEP)
        print("Target step:", TOTAL_STEPS)

    # ------------------------------------------------------------
    # TOKENIZER
    # ------------------------------------------------------------

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        use_fast=True
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    # ------------------------------------------------------------
    # DATASET AND SAMPLER
    # ------------------------------------------------------------

    train_dataset = MalwareDataset(
        TRAIN_FILE
    )

    sampler = DistributedSampler(
        train_dataset,
        num_replicas=WORLD_SIZE,
        rank=RANK,
        shuffle=True,
        seed=SEED,
        drop_last=False
    )

    sampler.set_epoch(0)

    collator = MalwareCollator(
        tokenizer.pad_token_id
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        collate_fn=collator,
        num_workers=0,
        pin_memory=True,
        drop_last=False
    )

    # ------------------------------------------------------------
    # VERIFY ORIGINAL TRAINING POSITION
    # ------------------------------------------------------------

    batches_per_epoch = len(train_loader)

    expected_batches = 8847

    assert batches_per_epoch == expected_batches, (
        f"Expected {expected_batches} batches per GPU, "
        f"found {batches_per_epoch}"
    )

    START_BATCH = (
        CHECKPOINT_STEP * GRAD_ACCUMULATION
    )

    assert START_BATCH < batches_per_epoch

    if RANK == 0:

        print("\nTraining dataset:", len(train_dataset))
        print("Batches per GPU:", batches_per_epoch)
        print("Skipping completed batches:", START_BATCH)
        print(
            "Remaining microbatches:",
            batches_per_epoch - START_BATCH
        )

    # ------------------------------------------------------------
    # LOAD 4-BIT BASE MODEL
    # ------------------------------------------------------------

    if RANK == 0:
        print("\nLoading 4-bit DeepSeek base model...")

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map={"": LOCAL_RANK},
        trust_remote_code=True
    )

    base_model.config.use_cache = False

    base_model = prepare_model_for_kbit_training(
        base_model
    )

    # ------------------------------------------------------------
    # LOAD TRAINABLE CHECKPOINT ADAPTER
    # ------------------------------------------------------------

    if RANK == 0:
        print("\nLoading checkpoint-300 LoRA adapter...")

    model = PeftModel.from_pretrained(
        base_model,
        CHECKPOINT_DIR,
        is_trainable=True
    )

    model.train()

    # ------------------------------------------------------------
    # DDP
    # ------------------------------------------------------------

    model = DDP(
        model,
        device_ids=[LOCAL_RANK],
        output_device=LOCAL_RANK,
        find_unused_parameters=False
    )

    # ------------------------------------------------------------
    # OPTIMIZER
    # ------------------------------------------------------------

    trainable_parameters = [
        p for p in model.parameters()
        if p.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        trainable_parameters,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    # ------------------------------------------------------------
    # SCHEDULER
    # ------------------------------------------------------------

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda
    )

    # ------------------------------------------------------------
    # RESTORE OPTIMIZER AND SCHEDULER
    # ------------------------------------------------------------

    if RANK == 0:
        print("\nRestoring optimizer and scheduler states...")

    checkpoint_state = torch.load(
        CHECKPOINT_DIR / "training_state.pt",
        map_location="cpu",
        weights_only=False
    )

    assert checkpoint_state["global_step"] == CHECKPOINT_STEP

    optimizer.load_state_dict(
        checkpoint_state["optimizer"]
    )

    scheduler.load_state_dict(
        checkpoint_state["scheduler"]
    )

    del checkpoint_state

    if RANK == 0:

        print("Optimizer restored.")
        print("Scheduler restored.")
        print("Current learning rate:", scheduler.get_last_lr()[0])

    # ------------------------------------------------------------
    # VERIFY OPTIMIZER PARAMETER STATE
    # ------------------------------------------------------------

    if RANK == 0:

        print(
            "Trainable LoRA parameters:",
            sum(p.numel() for p in trainable_parameters)
        )

    # ------------------------------------------------------------
    # TRAINING
    # ------------------------------------------------------------

    global_step = CHECKPOINT_STEP

    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    loss_count = 0

    model.train()

    if RANK == 0:

        print("\n" + "=" * 65)
        print("RESUMING TRAINING")
        print("=" * 65)

    for batch_index, batch in enumerate(train_loader):

        # Skip the batches already consumed before checkpoint 300.

        if batch_index < START_BATCH:
            continue

        # Determine original accumulation-group size.
        # The final group contains 15 batches, not 16.

        group_start = (
            batch_index // GRAD_ACCUMULATION
        ) * GRAD_ACCUMULATION

        group_size = min(
            GRAD_ACCUMULATION,
            batches_per_epoch - group_start
        )

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss / group_size

        loss.backward()

        running_loss += outputs.loss.detach().item()
        loss_count += 1

        is_accumulation_boundary = (
            (batch_index + 1) % GRAD_ACCUMULATION == 0
        )

        is_last_batch = (
            batch_index + 1 == batches_per_epoch
        )

        if is_accumulation_boundary or is_last_batch:

            torch.nn.utils.clip_grad_norm_(
                trainable_parameters,
                max_norm=1.0
            )

            optimizer.step()

            scheduler.step()

            optimizer.zero_grad(
                set_to_none=True
            )

            global_step += 1

            # Synchronize before checkpointing.

            dist.barrier()

            if RANK == 0:

                average_loss = (
                    running_loss / max(1, loss_count)
                )

                print(
                    f"Step {global_step}/{TOTAL_STEPS} | "
                    f"Average Loss: {average_loss:.6f} | "
                    f"LR: {scheduler.get_last_lr()[0]:.8f}",
                    flush=True
                )

                running_loss = 0.0
                loss_count = 0

            if global_step % 50 == 0:

                save_checkpoint(
                    model,
                    tokenizer,
                    optimizer,
                    scheduler,
                    global_step,
                    RANK
                )

                dist.barrier()

    # ------------------------------------------------------------
    # FINAL VERIFICATION
    # ------------------------------------------------------------

    assert global_step == TOTAL_STEPS, (
        f"Expected final step {TOTAL_STEPS}, "
        f"got {global_step}"
    )

    dist.barrier()

    # ------------------------------------------------------------
    # SAVE FINAL ADAPTER
    # ------------------------------------------------------------

    if RANK == 0:

        final_path = OUTPUT_DIR / "final_adapter"

        final_path.mkdir(
            parents=True,
            exist_ok=True
        )

        model.module.save_pretrained(
            final_path
        )

        tokenizer.save_pretrained(
            final_path
        )

        torch.save(
            {
                "global_step": global_step,
                "learning_rate": LEARNING_RATE,
                "gradient_accumulation_steps": GRAD_ACCUMULATION,
                "epochs_completed": 1,
                "validation_performed": False
            },
            OUTPUT_DIR / "training_summary.pt"
        )

        print("\n" + "=" * 65)
        print("RESUMED TRAINING COMPLETED")
        print("=" * 65)

        print("Final global step:", global_step)
        print("Final adapter:", final_path)

    dist.barrier()

    dist.destroy_process_group()


if __name__ == "__main__":
    train_worker()
'''

SCRIPT_PATH.write_text(
    script,
    encoding="utf-8"
)

print("\nDistributed training script created.")


# ================================================================
# 7. CHECK GPU AVAILABILITY
# ================================================================

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available.")

if torch.cuda.device_count() != NUM_GPUS:
    raise RuntimeError(
        f"Expected {NUM_GPUS} GPUs, "
        f"found {torch.cuda.device_count()}."
    )

print("\nAvailable GPUs:")

for i in range(torch.cuda.device_count()):

    print(
        f"GPU {i}:",
        torch.cuda.get_device_name(i)
    )


# ================================================================
# 8. LAUNCH TWO-GPU DISTRIBUTED RESUMPTION
# ================================================================

print("\nLaunching two-GPU training...")
print("Checkpoint: step 300")
print("Resume: step 301")
print("Target: step 553")
print("\nPlease do not interrupt the Kaggle session.")

command = [
    sys.executable,
    "-m",
    "torch.distributed.run",
    "--standalone",
    "--nnodes=1",
    "--nproc_per_node=2",
    str(SCRIPT_PATH)
]

result = subprocess.run(
    command,
    check=False
)

if result.returncode != 0:

    raise RuntimeError(
        f"Distributed training failed with exit code "
        f"{result.returncode}. Check the logs above."
    )


# ================================================================
# 9. FINAL OUTPUT VERIFICATION
# ================================================================

FINAL_ADAPTER = OUTPUT_DIR / "final_adapter"

if not FINAL_ADAPTER.exists():
    raise FileNotFoundError(
        "Final adapter folder was not created."
    )

required_final_files = [
    "adapter_config.json",
    "adapter_model.safetensors"
]

for filename in required_final_files:

    path = FINAL_ADAPTER / filename

    if not path.exists():
        raise FileNotFoundError(
            f"Missing final adapter file: {path}"
        )

print("\n" + "=" * 65)
print("CELL 2 COMPLETED SUCCESSFULLY")
print("=" * 65)

print("\nFinal adapter location:")
print(FINAL_ADAPTER)

print("\nCheckpoint location:")
print(OUTPUT_DIR)

print("\nYou can now download the final_adapter folder.")

DEEPSEEK CHECKPOINT RESUMPTION

Tokenized datasets found.
Training: /kaggle/working/deepseek_malware/train_tokenized.jsonl
Validation: /kaggle/working/deepseek_malware/validation_tokenized.jsonl
Testing: /kaggle/working/deepseek_malware/test_tokenized.jsonl

Copying checkpoint files...
Copied: adapter_config (3).json
Copied: adapter_model (6).safetensors
Copied: tokenizer (3).json
Copied: tokenizer_config (4).json
Copied: training_config (1).json
Copied: training_state.pt

Checkpoint verification successful.
Global step: 300
Optimizer state entries: 336
Scheduler state restored from checkpoint.

Output directory: /kaggle/working/deepseek_malware_resumed

Distributed training script created.

Available GPUs:
GPU 0: Tesla T4
GPU 1: Tesla T4

Launching two-GPU training...
Checkpoint: step 300
Resume: step 301
Target: step 553

Please do not interrupt the Kaggle session.



*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************



Distributed training initialized.
World size: 2
Starting from step: 300
Target step: 553


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!



Training dataset: 17693
Batches per GPU: 8847
Skipping completed batches: 4800
Remaining microbatches: 4047

Loading 4-bit DeepSeek base model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 219/219 [00:06<00:00, 35.44it/s]



Loading checkpoint-300 LoRA adapter...

Restoring optimizer and scheduler states...
Optimizer restored.
Scheduler restored.
Current learning rate: 9.42271880819367e-05
Trainable LoRA parameters: 14991360

RESUMING TRAINING


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

Step 301/553 | Average Loss: 0.085592 | LR: 0.00009385
Step 302/553 | Average Loss: 0.094952 | LR: 0.00009348
Step 303/553 | Average Loss: 0.076771 | LR: 0.00009311
Step 304/553 | Average Loss: 0.070208 | LR: 0.00009274
Step 305/553 | Average Loss: 0.061451 | LR: 0.00009236
Step 306/553 | Average Loss: 0.083069 | LR: 0.00009199
Step 307/553 | Average Loss: 0.099955 | LR: 0.00009162
Step 308/553 | Average Loss: 0.077902 | LR: 0.00009125
Step 309/553 | Average Loss: 0.125251 | LR: 0.00009088
Step 310/553 | Average Loss: 0.068373 | LR: 0.00009050
Step 311/553 | Average Loss: 0.130059 | LR: 0.00009013
Step 312/553 | Average Loss: 0.099572 | LR: 0.00008976
Step 313/553 | Average Loss: 0.064129 | LR: 0.00008939
Step 314/553 | Average Loss: 0.084353 | LR: 0.00008901
Step 315/553 | Average Loss: 0.078278 | LR: 0.00008864
Step 316/553 | Average Loss: 0.059874 | LR: 0.00008827
Step 317/553 | Average Loss: 0.055136 | LR: 0.00008790
Step 318/553 | Average Loss: 0.086668 | LR: 0.00008752
Step 319/5